# FashionMNIST_Keras_SGD

이 노트북은 발표용 재실험을 위해 **한 가지 조건만 명확하게 바꾸고**,  
학습 과정과 결과를 동일한 형식으로 저장하도록 구성했습니다.

## CNN 흐름과 코드 위치
- **[1] 데이터 입력**
- **[2] 특징 추출 - Conv Block 1**
- **[3] 특징 추출 - Conv Block 2**
- **[4] 추가 특징 추출 / 분류**
- **[5] Optimizer 설정**
- **[6] 학습**
- **[7] 성능 평가**
- **[8] 결과 저장**

> Keras와 PyTorch 모두 같은 번호 체계를 사용해 PPT에서 코드 위치를 비교하기 쉽게 만들었습니다.

In [ ]:
# 공통 설정
import os, time, random, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
EPOCHS = 10
BATCH_SIZE = 64
LEARNING_RATE = 0.01
MOMENTUM = 0.9
EXPERIMENT_NAME = "FashionMNIST_Keras_SGD"

OUTPUT_DIR = Path("/content/experiment_outputs") / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("Experiment:", EXPERIMENT_NAME)
print("Output:", OUTPUT_DIR)

## [1] 데이터 입력

Fashion-MNIST의 공식 Train 60,000장을 고정 Seed로 다시 나눠  
**Train 48,000 / Validation 12,000**으로 사용하고, 공식 Test 10,000장을 최종 평가에 사용합니다.

In [ ]:
# [1] 데이터 입력
(x_full, y_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

train_idx, val_idx = train_test_split(
    np.arange(len(y_full)),
    test_size=12000,
    random_state=SEED,
    stratify=y_full
)

x_train, y_train = x_full[train_idx], y_full[train_idx]
x_val, y_val = x_full[val_idx], y_full[val_idx]

# Keras Conv2D 입력 형태: (N, H, W, C)
x_train = x_train[..., np.newaxis]
x_val = x_val[..., np.newaxis]
x_test = x_test[..., np.newaxis]

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

## [2]~[4] CNN 모델 정의

PPT에서 설명한 흐름과 직접 대응합니다.

`Input → Conv → BatchNorm → ReLU → Pool → Conv → BatchNorm → ReLU → Pool → Flatten → Dense → Output`

In [ ]:
model = keras.Sequential([
    # [1] 입력: 28×28 흑백 이미지
    layers.Input(shape=(28, 28, 1)),
    layers.Rescaling(1.0 / 255.0),

    # [2] 특징 추출 - Conv Block 1
    # Conv → BatchNorm → ReLU → Pooling
    layers.Conv2D(32, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D(2),

    # [3] 특징 추출 - Conv Block 2
    # 더 복잡한 특징을 64개의 Feature Map으로 추출
    layers.Conv2D(64, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D(2),

    # [4] 분류
    # Flatten → Dense(FC) → ReLU → Dropout → Output
    layers.Flatten(),
    layers.Dense(128),
    layers.ReLU(),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])

In [ ]:
model.summary()
parameter_count = model.count_params()
print("Total parameters:", f"{parameter_count:,}")

## [5] Optimizer 설정

- Framework: **Keras**
- Optimizer: **SGD**
- Learning Rate: **0.01**
- Momentum: **0.9**

Keras는 `compile()`에서 Loss / Optimizer / Metric을 연결합니다.

In [ ]:
# [5] Optimizer 설정
optimizer = keras.optimizers.SGD(learning_rate=LEARNING_RATE, momentum=MOMENTUM)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Best Validation Loss 모델을 별도로 저장
BEST_MODEL_PATH = OUTPUT_DIR / "best_val_loss.keras"
checkpoint_cb = keras.callbacks.ModelCheckpoint(
    filepath=BEST_MODEL_PATH,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

## [6] 학습

Keras의 특징이 잘 보이도록 학습 부분을 별도 셀로 분리했습니다.  
`model.fit()` 한 번이 Epoch 반복, forward, loss 계산, 역전파, parameter update를 내부에서 처리합니다.

In [ ]:
# [6] 학습
# Keras는 반복 학습 루프를 model.fit() 내부에서 처리합니다.
start_time = time.time()
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[checkpoint_cb],
    verbose=1
)
training_seconds = time.time() - start_time

## [7] 성능 평가

최종 Epoch 모델과 **Best Validation Loss 모델**을 모두 평가합니다.  
Accuracy뿐 아니라 Precision / Recall / F1 / Confusion Matrix도 저장합니다.

In [ ]:
# History DataFrame
history_df = pd.DataFrame(history.history)
history_df.index = np.arange(1, len(history_df) + 1)
history_df.index.name = "epoch"

best_epoch = int(history_df["val_loss"].idxmin())
best_val_loss = float(history_df.loc[best_epoch, "val_loss"])
best_val_accuracy = float(history_df["val_accuracy"].max())

# 최종 Epoch 모델
final_probs = model.predict(x_test, batch_size=BATCH_SIZE, verbose=0)
final_pred = np.argmax(final_probs, axis=1)

# Best Validation Loss 모델
best_model = keras.models.load_model(BEST_MODEL_PATH)
best_probs = best_model.predict(x_test, batch_size=BATCH_SIZE, verbose=0)
best_pred = np.argmax(best_probs, axis=1)

def evaluate_predictions(y_true, y_pred, prefix):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    report = pd.DataFrame(
        classification_report(
            y_true, y_pred, target_names=CLASS_NAMES,
            output_dict=True, zero_division=0
        )
    ).T
    report.to_csv(OUTPUT_DIR / f"{prefix}_classification_report.csv", encoding="utf-8-sig")
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, xticks_rotation=45, values_format="d"
    )
    ax.set_title(f"{EXPERIMENT_NAME} - {prefix} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{prefix}_confusion_matrix.png", dpi=160, bbox_inches="tight")
    plt.show()

    return {
        f"{prefix}_test_accuracy": acc,
        f"{prefix}_macro_precision": p,
        f"{prefix}_macro_recall": r,
        f"{prefix}_macro_f1": f1
    }

final_metrics = evaluate_predictions(y_test, final_pred, "final")
best_metrics = evaluate_predictions(y_test, best_pred, "best")

In [ ]:
# Accuracy / Loss 학습곡선
plt.figure(figsize=(9, 5))
plt.plot(history_df.index, history_df["accuracy"], label="Train Accuracy")
plt.plot(history_df.index, history_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(f"{EXPERIMENT_NAME} - Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "accuracy_curve.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(history_df.index, history_df["loss"], label="Train Loss")
plt.plot(history_df.index, history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{EXPERIMENT_NAME} - Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "loss_curve.png", dpi=160, bbox_inches="tight")
plt.show()

## [8] 결과 저장

모든 실험이 동일한 컬럼을 사용하도록 Summary CSV를 저장합니다.  
나중에 20개 실험 결과를 한 번에 합쳐 막대그래프나 표로 비교할 수 있습니다.

In [ ]:
history_df.to_csv(OUTPUT_DIR / "history.csv", encoding="utf-8-sig")

summary = {
    "experiment": EXPERIMENT_NAME,
    "dataset": "Fashion-MNIST",
    "framework": "Keras",
    "optimizer": "SGD",
    "model_version": "CommonCNN",
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "momentum": MOMENTUM if "SGD" == "SGD" else np.nan,
    "parameter_count": parameter_count,
    "training_seconds": training_seconds,
    "best_epoch_by_val_loss": best_epoch,
    "best_val_loss": best_val_loss,
    "best_val_accuracy": best_val_accuracy,
    "final_train_accuracy": float(history_df.iloc[-1]["accuracy"]),
    "final_val_accuracy": float(history_df.iloc[-1]["val_accuracy"]),
    "final_train_val_gap": float(history_df.iloc[-1]["accuracy"] - history_df.iloc[-1]["val_accuracy"]),
    "model_definition_code_lines": 17,
    "training_code_lines": 10,
    **final_metrics,
    **best_metrics
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False, encoding="utf-8-sig")
display(summary_df.T)

# 결과 폴더 압축
shutil_base = str(OUTPUT_DIR)
import shutil
zip_path = shutil.make_archive(shutil_base, "zip", root_dir=OUTPUT_DIR)
print("Saved:", zip_path)

## [9] 결과 ZIP 자동 다운로드

Google Colab에서 실행 중이라면, 저장된 실험 결과 ZIP을 브라우저로 자동 다운로드합니다.

In [ ]:
# ------------------------------------------------------------
# [9] Colab에서 결과 ZIP 자동 다운로드
# 학습/평가/저장이 끝난 뒤 생성된 ZIP 파일을 브라우저로 다운로드합니다.
# ------------------------------------------------------------
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Google Colab 환경이 아니므로 자동 다운로드를 실행하지 않았습니다.")
    print("결과 ZIP 경로:", zip_path)